In [83]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu, norm

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style, FS_ANNOT_LG, FS_ANNOT_SM
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.validation import lobo_mmd

LOBO = config.LOBO_MIXED_DIR
FIG = LOBO / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)

In [84]:
FDR_Q, Z_CAP, MIN_N = 0.05, 10.0, 10
EXCLUDE_BATCHES = ('Chang et al._Batch_3', 'Gardella et al._Batch_5')
def _p_to_asterisk(p):
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def _short_batch_label(batch):
    name, _, num = batch.replace(' et al.', '').partition('_Batch_')
    return f'{name}_{num}' if num else name


def bh_q(p):
    p = np.asarray(p, dtype=float)
    m = len(p)
    o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * m / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m)
    out[o] = np.clip(q, 0, 1)
    return out


def rkhs_distance_stats(shash, ood_filter):
    rows, raw = [], {}
    for p in sorted(LOBO.glob('*/meta.json')):
        meta = json.load(open(p))
        b = meta['batch_id']
        if b in EXCLUDE_BATCHES:
            continue
        Z = np.load(p.parent / ('Z_test_shash.npy' if shash else 'Z_test.npy')).astype(np.float32)
        is_hc = np.array(meta['test_is_hc'])
        if ood_filter:
            keep = np.load(p.parent / 'ood_mask.npy')
            Z, is_hc = Z[keep], is_hc[keep]
        n_hc, n_dis = int(is_hc.sum()), int((~is_hc).sum())
        if min(n_hc, n_dis) < MIN_N:
            continue
        Z = np.nan_to_num(Z, nan=0.0, posinf=Z_CAP, neginf=-Z_CAP)
        d_hc, d_dis, _ = lobo_mmd._ref_centroid_direction(Z, is_hc)
        u, pu = mannwhitneyu(d_dis, d_hc, alternative='greater')
        rows.append(dict(batch=b, n_hc=n_hc, n_dis=n_dis, auc=u / (n_dis * n_hc), mw_p=pu))
        raw[b] = dict(d_hc=d_hc, d_dis=d_dis)
    stats = pd.DataFrame(rows).sort_values('auc', ascending=False)
    stats['bh_q'] = bh_q(stats['mw_p'].values)
    return stats, raw


def plot_mmd_direction(stats, raw, out_path, p_col='mw_p'):
    from matplotlib.patches import Patch
    from scipy.stats import gaussian_kde
    d = stats.sort_values('auc', ascending=False).reset_index(drop=True)
    n_batches = len(d)

    fig, axes = plt.subplots(n_batches, 1, figsize=(7, 1.2 * n_batches), sharex=True)
    if n_batches == 1:
        axes = [axes]

    color_hc = '#A4AFB8'
    color_dis = '#be3b3b'

    for ax, (_, row) in zip(axes, d.iterrows()):
        b = row['batch']
        r = raw[b]

        d_hc = np.asarray(r['d_hc'])
        d_dis = np.asarray(r['d_dis'])

        x_min = min(d_hc.min(), d_dis.min())
        x_max = max(d_hc.max(), d_dis.max())
        x_margin = (x_max - x_min) * 0.2
        x_grid = np.linspace(x_min - x_margin, x_max + x_margin, 300)

        kde_hc = gaussian_kde(d_hc)(x_grid)
        kde_dis = gaussian_kde(d_dis)(x_grid)

        ax.plot(x_grid, kde_hc, color=color_hc, lw=1.5)
        ax.fill_between(x_grid, kde_hc, color=color_hc, alpha=0.35)

        ax.plot(x_grid, kde_dis, color=color_dis, lw=1.5)
        ax.fill_between(x_grid, kde_dis, color=color_dis, alpha=0.35)

        mean_hc = d_hc.mean()
        mean_dis = d_dis.mean()
        max_y = max(kde_hc.max(), kde_dis.max())

        ax.axvline(mean_hc, color=color_hc, linestyle='--', lw=1.5, zorder=3)
        ax.axvline(mean_dis, color=color_dis, linestyle='--', lw=1.5, zorder=3)

        y_bar = max_y * 1.15
        ax.annotate('', xy=(mean_hc, y_bar), xytext=(mean_dis, y_bar),
                    arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))

        p_val = row.get(p_col, 1.0)
        asterisk = _p_to_asterisk(p_val)
        delta = abs(mean_dis - mean_hc)

        mid_x = (mean_hc + mean_dis) / 2
        text_str = f"Δ={delta:.3f} ({asterisk})"

        ax.text(mid_x, y_bar + max_y * 0.08, text_str,
                ha='center', va='bottom', fontweight='bold', color='black')

        label = _short_batch_label(b)
        ax.set_ylabel(label, rotation=0, ha='right', va='center')
        ax.set_ylim(0, max_y * 1.55)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, axis='x', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Kernel embedding distance from HC reference')

    legend_elements = [
        Patch(facecolor=color_hc, edgecolor=color_hc, alpha=0.5, label='Held-out HC'),
        Patch(facecolor=color_dis, edgecolor=color_dis, alpha=0.5, label='Disease')
    ]
    axes[0].legend(handles=legend_elements, loc='upper right', frameon=False)

    plt.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', dpi=300)
    return fig


def build_nsig(shash, ood_filter, cache_name):
    path = LOBO / cache_name
    if path.exists():
        return pd.read_csv(path)
    rows = []
    for p in sorted(LOBO.glob('*/meta.json')):
        m = json.load(open(p))
        Z = np.load(p.parent / ('Z_test_shash.npy' if shash else 'Z_test.npy')).astype(np.float32)
        names, is_hc = np.array(m['test_names']), np.array(m['test_is_hc'])
        if ood_filter:
            keep = np.load(p.parent / 'ood_mask.npy')
            Z, names, is_hc = Z[keep], names[keep], is_hc[keep]
        Z = np.clip(np.nan_to_num(Z, nan=0.0), -Z_CAP, Z_CAP)
        ns = [int(bh_fdr_reject(r, q=FDR_Q).sum()) for r in 2 * norm.sf(np.abs(Z))]
        rows += [(m['batch_id'], nm, bool(hc), v) for nm, hc, v in zip(names, is_hc, ns)]
    df = pd.DataFrame(rows, columns=['batch', 'sample', 'is_hc', 'n_sig'])
    df.to_csv(path, index=False)
    return df


def nsig_batch_stats(nsig, cache_name):
    rows = []
    for b, g in nsig[~nsig['batch'].isin(EXCLUDE_BATCHES)].groupby('batch'):
        h = g.loc[g['is_hc'], 'n_sig'].values
        d = g.loc[~g['is_hc'], 'n_sig'].values
        if min(len(h), len(d)) < MIN_N:
            continue
        u, pu = mannwhitneyu(d, h, alternative='greater')
        rows.append(dict(batch=b, n_hc=len(h), n_dis=len(d), hc_med=np.median(h), dis_med=np.median(d),
                         auc=u / (len(d) * len(h)), mw_p=pu))
    stats = pd.DataFrame(rows).sort_values('auc', ascending=False)
    stats['bh_q'] = bh_q(stats['mw_p'].values)
    stats.to_csv(LOBO / cache_name, index=False)
    return stats


def plot_nsig_box(nsig, stats, title, out_path):
    GROUPS = ['Disease', 'HC (held-out)']
    order = stats.sort_values('dis_med', ascending=False)['batch'].tolist()
    plot = nsig[nsig['batch'].isin(order)].copy()
    plot['group'] = np.where(plot['is_hc'], GROUPS[1], GROUPS[0])
    plot['x'] = np.log10(plot['n_sig'] + 1)
    q_of = dict(zip(stats['batch'], stats['bh_q']))

    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(order) + 1.8))
    sns.boxplot(data=plot, y='batch', x='x', hue='group', order=order, hue_order=GROUPS,
                   palette={GROUPS[0]: "#be3b3b", GROUPS[1]: '#c8cdd2'}, orient='h', linecolor='black',
                   linewidth=0.8, width=0.65, ax=ax, showfliers=False)

    TICKS = [0, 1, 3, 10, 30, 100, 300, 1000, 10000]
    ax.set_xticks(np.log10(np.array(TICKS) + 1.0))
    ax.set_xticklabels(TICKS)
    x_max = plot['x'].max()
    ax.set_xlim(-0.12, x_max + 0.75)

    for i, b in enumerate(order):
        x_br = plot['x'].max() + 0.16
        ax.plot([x_br, x_br + 0.07, x_br + 0.07, x_br], [i - 0.21, i - 0.21, i + 0.21, i + 0.21],
                color='black', lw=0.9, clip_on=False)
        ax.text(x_br + 0.12, i, _p_to_asterisk(q_of[b]) or 'ns', ha='left', va='center',
                 clip_on=False)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([_short_batch_label(b) for b in order])
    ax.set_xlabel('BH-significant genes per sample (q=0.05)')
    ax.set_ylabel('Batch (held-out)')
    ax.set_title(title, loc='left')
    h, l = ax.get_legend_handles_labels()
    ax.get_legend().remove()

    plt.tight_layout()
    fig.legend(h[:2], l[:2], frameon=False,
               loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=2)
    fig.savefig(out_path, bbox_inches='tight')
    return fig

In [85]:
from MixedEffectsModeling.validation.lobo_engine import compute_ood

compute_ood()

Chang et al._Batch_1                     n_test=  30  removed_OOD=  9 (30.0%)
Chang et al._Batch_11                    n_test=  35  removed_OOD=  1 (2.9%)
Chang et al._Batch_2                     n_test=  37  removed_OOD=  0 (0.0%)
Chang et al._Batch_3                     n_test=  30  removed_OOD=  1 (3.3%)
Chang et al._Batch_4                     n_test=  33  removed_OOD=  0 (0.0%)
Chang et al._Batch_5                     n_test=  25  removed_OOD=  0 (0.0%)
Chen et al._Batch_2                      n_test=  63  removed_OOD=  8 (12.7%)
Gardella et al._Batch_1                  n_test=  30  removed_OOD=  0 (0.0%)
Gardella et al._Batch_3                  n_test=  14  removed_OOD=  1 (7.1%)
Gardella et al._Batch_4                  n_test=  46  removed_OOD=  1 (2.2%)
Gardella et al._Batch_5                  n_test=  32  removed_OOD=  0 (0.0%)
Gardella et al._Batch_7                  n_test=  17  removed_OOD=  0 (0.0%)
Moore et al._Batch_1                     n_test= 244  removed_OOD= 16 (6.6

## Variant 1 -- raw (no SHASH, no OOD filter)

In [86]:

dist_raw, raw_raw = rkhs_distance_stats(shash=False, ood_filter=False)
print('RKHS kernel-embedding distance, disease vs held-out HC (no permutation):')
print(dist_raw.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
plot_mmd_direction(dist_raw, raw_raw, FIG / 'mmd_direction_raw.png')

nsig_raw = build_nsig(shash=False, ood_filter=False, cache_name='lobo_nsig_raw.csv')
stats_raw = nsig_batch_stats(nsig_raw, cache_name='batch_matched_nsig_stats_raw.csv')
print(f"\nn_sig: {len(stats_raw)} batches, BH q<{FDR_Q}: {(stats_raw['bh_q'] < FDR_Q).sum()}")
print(stats_raw.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

plot_nsig_box(nsig_raw, stats_raw, 'Raw (no SHASH, no OOD filter)',
              FIG / 'batch_matched_nsig_raw.png')
plt.close('all')

RKHS kernel-embedding distance, disease vs held-out HC (no permutation):
                          batch  n_hc  n_dis    auc   mw_p   bh_q
          Ward Z et al._Batch_1   116    224 0.7567 0.0000 0.0000
          Chang et al._Batch_11    12     23 0.7500 0.0086 0.0176
       Moufarrej et al._Batch_4    47     39 0.6748 0.0028 0.0138
Roskams-Hieter B et al._Batch_2    27     58 0.6609 0.0088 0.0176
       Moufarrej et al._Batch_1    54     23 0.6433 0.0241 0.0402
            Chen et al._Batch_2    31     32 0.6361 0.0322 0.0460
           Moore et al._Batch_1    71    173 0.5969 0.0087 0.0176
        Gardella et al._Batch_4    18     28 0.5734 0.2057 0.2571
           Chang et al._Batch_5    14     11 0.5260 0.4240 0.4711
           Chang et al._Batch_1    13     17 0.3213 0.9529 0.9529

n_sig: 10 batches, BH q<0.05: 9
                          batch  n_hc  n_dis  hc_med  dis_med    auc   mw_p   bh_q
Roskams-Hieter B et al._Batch_2    27     58  1.0000  28.5000 0.9033 0.0000 0.0000
  

## Variant 2 -- OOD filter only (no SHASH)

In [87]:
dist_ood, raw_ood = rkhs_distance_stats(shash=False, ood_filter=True)
print('RKHS kernel-embedding distance, disease vs held-out HC (no permutation):')
print(dist_ood.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
plot_mmd_direction(dist_ood, raw_ood, FIG / 'mmd_direction_ood.png')

nsig_ood = build_nsig(shash=False, ood_filter=True, cache_name='lobo_nsig_oodonly.csv')
stats_ood = nsig_batch_stats(nsig_ood, cache_name='batch_matched_nsig_stats_oodonly.csv')
print(f"\nn_sig: {len(stats_ood)} batches, BH q<{FDR_Q}: {(stats_ood['bh_q'] < FDR_Q).sum()}")
print(stats_ood.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

plot_nsig_box(nsig_ood, stats_ood, 'OOD filter only (no SHASH)',
              FIG / 'batch_matched_nsig_ood.png')
plt.close('all')

RKHS kernel-embedding distance, disease vs held-out HC (no permutation):
                          batch  n_hc  n_dis    auc   mw_p   bh_q
          Ward Z et al._Batch_1   105    190 0.7372 0.0000 0.0000
          Chang et al._Batch_11    12     22 0.7348 0.0133 0.0221
       Moufarrej et al._Batch_1    47     18 0.6761 0.0147 0.0221
            Chen et al._Batch_2    27     28 0.6627 0.0196 0.0252
       Moufarrej et al._Batch_4    42     30 0.6627 0.0098 0.0221
Roskams-Hieter B et al._Batch_2    26     57 0.6532 0.0131 0.0221
           Moore et al._Batch_1    67    161 0.5947 0.0122 0.0221
        Gardella et al._Batch_4    17     28 0.5714 0.2164 0.2435
           Chang et al._Batch_5    14     11 0.5260 0.4240 0.4240

n_sig: 9 batches, BH q<0.05: 9
                          batch  n_hc  n_dis  hc_med  dis_med    auc   mw_p   bh_q
Roskams-Hieter B et al._Batch_2    26     57  1.0000  28.0000 0.9005 0.0000 0.0000
          Ward Z et al._Batch_1   105    190  0.0000   6.0000 0.8827 

## Variant 3 -- SHASH + OOD filter

In [88]:
# Variant 3: SHASH + OOD filter -- the variant used elsewhere in this notebook's
# text (per-batch SHASH fit on train-fold in-sample Z, OOD-extrapolating samples
# dropped).
dist_shash_ood, raw_shash_ood = rkhs_distance_stats(shash=True, ood_filter=True)
print('RKHS kernel-embedding distance, disease vs held-out HC (no permutation):')
print(dist_shash_ood.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
plot_mmd_direction(dist_shash_ood, raw_shash_ood, FIG / 'mmd_direction_shash.png')

nsig_shash_ood = build_nsig(shash=True, ood_filter=True, cache_name='lobo_nsig_shashood.csv')
stats_shash_ood = nsig_batch_stats(nsig_shash_ood, cache_name='batch_matched_nsig_stats_shashood.csv')
print(f"\nn_sig: {len(stats_shash_ood)} batches, BH q<{FDR_Q}: {(stats_shash_ood['bh_q'] < FDR_Q).sum()}")
print(stats_shash_ood.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

plot_nsig_box(nsig_shash_ood, stats_shash_ood, 'SHASH + OOD filter',
              FIG / 'batch_matched_nsig_shashood.png')
plt.close('all')

RKHS kernel-embedding distance, disease vs held-out HC (no permutation):
                          batch  n_hc  n_dis    auc   mw_p   bh_q
          Ward Z et al._Batch_1   105    190 0.7331 0.0000 0.0000
          Chang et al._Batch_11    12     22 0.7273 0.0160 0.0288
       Moufarrej et al._Batch_1    47     18 0.6619 0.0227 0.0340
Roskams-Hieter B et al._Batch_2    26     57 0.6552 0.0121 0.0288
       Moufarrej et al._Batch_4    42     30 0.6516 0.0148 0.0288
            Chen et al._Batch_2    27     28 0.6508 0.0280 0.0360
           Moore et al._Batch_1    67    161 0.5926 0.0139 0.0288
        Gardella et al._Batch_4    17     28 0.5756 0.2030 0.2283
           Chang et al._Batch_5    14     11 0.5260 0.4240 0.4240

n_sig: 9 batches, BH q<0.05: 9
                          batch  n_hc  n_dis  hc_med  dis_med    auc   mw_p   bh_q
Roskams-Hieter B et al._Batch_2    26     57  1.0000  28.0000 0.9035 0.0000 0.0000
        Gardella et al._Batch_4    17     28  0.0000   1.5000 0.8992 